In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
SERVER   = r'localhost'   
TRUSTED  = "Trusted_Connection=yes;TrustServerCertificate=yes;"
sys_params = urllib.parse.quote_plus(f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER};DATABASE=master;{TRUSTED}")
sys_engine = create_engine(f"mssql+pyodbc:///?odbc_connect={sys_params}", isolation_level="AUTOCOMMIT")
print("Checking database status...")
with sys_engine.connect() as conn:
    conn.execute(text("IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = 'ipl_db') CREATE DATABASE ipl_db;"))
    print(" Database 'ipl_db' is ready and active!")
params = urllib.parse.quote_plus(f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER};DATABASE=ipl_db;{TRUSTED}")
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)
print("\nReading clean CSV files...")
df_matches = pd.read_csv('../data/matches_clean.csv')
df_del     = pd.read_csv('../data/deliveries_featured.csv')
batsman_stats = pd.read_csv('../data/batsman_match_stats.csv')
bowler_stats  = pd.read_csv('../data/bowler_match_stats.csv')
phase_summary = pd.read_csv('../data/phase_summary.csv')
print("\nLoading tables into ipl_db...")
df_matches.to_sql('matches', engine, if_exists='replace', index=False, chunksize=500)
print(f" matches → {len(df_matches):,} rows loaded.")
df_del.to_sql('deliveries', engine, if_exists='replace', index=False, chunksize=1000)
print(f" deliveries → {len(df_del):,} rows loaded.")
batsman_stats.to_sql('batsman_stats', engine, if_exists='replace', index=False)
print(f" batsman_stats → {len(batsman_stats):,} rows loaded.")
bowler_stats.to_sql('bowler_stats', engine, if_exists='replace', index=False)
print(f" bowler_stats → {len(bowler_stats):,} rows loaded.")
phase_summary.to_sql('phase_summary', engine, if_exists='replace', index=False)
print(f" phase_summary → {len(phase_summary):,} rows loaded.")
print("\n ALL TABLES LOADED SUCCESSFULY! Data pipeline complete.")

Checking database status...
 Database 'ipl_db' is ready and active!

Reading clean CSV files...

Loading tables into ipl_db...
 matches → 1,090 rows loaded.
 deliveries → 260,920 rows loaded.
 batsman_stats → 16,515 rows loaded.
 bowler_stats → 12,978 rows loaded.
 phase_summary → 3 rows loaded.

 ALL TABLES LOADED SUCCESSFULY! Data pipeline complete.


In [ ]:
import os
QUERIES = """

-- IPL ANALYTICS — SQL QUERIES
-- Database: ipl_db

-- QUERY 1: Top Batsman Per Season Using RANK()
WITH season_runs AS (
    SELECT
        season,
        batter,
        SUM(batsman_runs)  AS total_runs,
        COUNT(*)           AS balls_faced,
        ROUND(CAST(SUM(batsman_runs) AS FLOAT) / NULLIF(COUNT(*), 0) * 100, 2) AS strike_rate,
        RANK() OVER (PARTITION BY season ORDER BY SUM(batsman_runs) DESC) AS rnk
    FROM deliveries
    GROUP BY season, batter
)
SELECT season, batter AS top_batsman, total_runs, balls_faced, strike_rate
FROM season_runs WHERE rnk = 1 ORDER BY season;

-- QUERY 2: Season-over-Season Total Runs Growth Using LAG()
WITH season_totals AS (
    SELECT season, SUM(total_runs) AS total_runs FROM deliveries GROUP BY season
)
SELECT season, total_runs, LAG(total_runs) OVER (ORDER BY season) AS prev_season_runs,
    ROUND((CAST(total_runs AS FLOAT) - LAG(total_runs) OVER (ORDER BY season)) * 100.0 / NULLIF(LAG(total_runs) OVER (ORDER BY season), 0), 2) AS yoy_growth_pct
FROM season_totals ORDER BY season;

-- QUERY 3: Best Death-Over Bowlers (Economy < 8.5)
WITH death_stats AS (
    SELECT bowler, SUM(total_runs) AS runs_conceded,
        SUM(CASE WHEN extras_type NOT IN ('wides','noballs') OR extras_type IS NULL THEN 1 ELSE 0 END) AS legal_balls,
        SUM(CASE WHEN player_dismissed <> 'not_out' THEN 1 ELSE 0 END) AS wickets
    FROM deliveries WHERE phase = 'Death' GROUP BY bowler
    HAVING SUM(CASE WHEN extras_type NOT IN ('wides','noballs') OR extras_type IS NULL THEN 1 ELSE 0 END) >= 120
)
SELECT bowler, runs_conceded, ROUND(CAST(legal_balls AS FLOAT) / 6, 1) AS overs_bowled, wickets,
    ROUND(CAST(runs_conceded AS FLOAT) / NULLIF(CAST(legal_balls AS FLOAT) / 6, 0), 2) AS economy_rate
FROM death_stats WHERE ROUND(CAST(runs_conceded AS FLOAT) / NULLIF(CAST(legal_balls AS FLOAT) / 6, 0), 2) < 8.5
ORDER BY economy_rate;

-- QUERY 4: Team Win % — Bat First vs Chase
SELECT toss_decision, COUNT(*) AS total_matches,
    SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS toss_wins,
    ROUND(CAST(SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS FLOAT) / COUNT(*) * 100, 2) AS win_pct
FROM matches GROUP BY toss_decision ORDER BY win_pct DESC;

-- QUERY 5: Running Total of Runs Per Team Per Season
WITH team_season_runs AS (
    SELECT batting_team, season, SUM(batsman_runs) AS season_runs FROM deliveries GROUP BY batting_team, season
)
SELECT batting_team, season, season_runs,
    SUM(season_runs) OVER (PARTITION BY batting_team ORDER BY season ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_runs
FROM team_season_runs ORDER BY batting_team, season;

-- QUERY 6: Phase-wise Run Rate + Wicket Rate Summary
SELECT phase, SUM(total_runs) AS total_runs,
    SUM(CASE WHEN player_dismissed <> 'not_out' THEN 1 ELSE 0 END) AS total_wickets, COUNT(*) AS total_balls,
    ROUND(CAST(SUM(total_runs) AS FLOAT) / NULLIF(COUNT(*), 0) * 6, 2) AS run_rate,
    ROUND(CAST(SUM(CASE WHEN player_dismissed <> 'not_out' THEN 1 ELSE 0 END) AS FLOAT) / NULLIF(COUNT(*), 0) * 6, 4) AS wickets_per_over,
    ROUND(CAST(SUM(total_runs) AS FLOAT) / NULLIF(SUM(CASE WHEN player_dismissed <> 'not_out' THEN 1 ELSE 0 END), 0), 2) AS runs_per_wicket
FROM deliveries GROUP BY phase
ORDER BY CASE phase WHEN 'Powerplay' THEN 1 WHEN 'Middle' THEN 2 WHEN 'Death' THEN 3 END;
"""

os.makedirs('../sql', exist_ok=True)
with open('../sql/queries.sql', 'w', encoding='utf-8') as f:
    f.write(QUERIES)

print("\n SQL queries successfully saved to sql/queries.sql")
print(" You can check your VS Code sidebar to see the file!")


 SQL queries successfully saved to sql/queries.sql
  You can check your VS Code sidebar to see the file!
